# Cliente MCP con LangGraph

Agente ReAct que se conecta a `servidor_mcp.py` por stdio, descubre las herramientas y las usa en ciclos de razonamiento.

Ejecute las celdas en orden. Requiere Ollama (`qwen2.5:3b`) y el kernel `.venv`.

## Configuración del cliente MCP

In [ ]:
# MultiServerMCPClient: connects to one or more MCP servers and exposes their
# tools, resources, and prompts to the LangChain / LangGraph ecosystem.
from langchain_mcp_adapters.client import MultiServerMCPClient
from pathlib import Path
import sys

# resolve() converts the relative path to an absolute one so the server can be
# spawned from any working directory without a FileNotFoundError.
# sys.executable uses this notebook's venv Python (required on Windows).
SERVER_PATH = str(Path("servidor_mcp.py").resolve())

# Each key in server_config is the logical name used later as the session key.
# The value tells MultiServerMCPClient how to launch and talk to the server.
server_config = {
    "agents-tutorial": {
        "command": sys.executable,  # executable used to spawn the subprocess
        "args":    [SERVER_PATH],   # script passed as the first argument
        "transport": "stdio",       # MCP wire protocol runs over stdin/stdout
    }
}

In [ ]:
print(f"Server path : {SERVER_PATH}")
print(f"Transport   : stdio — server runs as a subprocess over stdin/stdout")
print(f"Server name : agents-tutorial")

### Instancia del cliente y prueba de conexión

In [ ]:
# Construction is lazy — no subprocess is spawned here.
# Connections are established on the first get_tools() call
# or when a client.session() context block is entered.
client = MultiServerMCPClient(server_config)

In [ ]:
# client.session(name) is an async context manager:
#   __aenter__ — spawns the subprocess and completes the MCP initialization handshake.
#   __aexit__  — sends a shutdown notification and terminates the process cleanly.
# The session (and the running server) only exist inside this block.
async with client.session("agents-tutorial") as session:
    print("Session open: agents-tutorial")

    # list_tools() sends a ListTools request over the MCP wire protocol and
    # returns a ListToolsResult whose .tools attribute holds the raw tool definitions.
    raw = await session.list_tools()
    print(f"\nServer exposes {len(raw.tools)} tools:")
    for t in raw.tools:
        print(f"  * {t.name}")

## Descubrimiento dinámico de herramientas

In [ ]:
client = MultiServerMCPClient(server_config)

# get_tools() queries list_tools on every configured server and converts each
# MCP tool schema into a LangChain BaseTool — no manual @tool definitions needed.
# This is dynamic tool discovery: the client adapts automatically to whatever
# tools the server exposes, even if new tools are added in a later server version.
tools = await client.get_tools()

In [ ]:
print(f"Discovered {len(tools)} LangChain-compatible tools:\n")
for tool in tools:
    # args_schema exposes the JSON Schema dict sent to the LLM when bind_tools() is called.
    # 'properties' lists the parameters the model must fill in for each tool call.
    props = list(tool.args_schema.get('properties', {}).keys())
    print(f"==========  {tool.name}  ==========")
    print(f"description : {tool.description}")
    print(f"parameters  : {props}")
    print()

## Lectura de recursos y prompts del servidor

In [ ]:
# A separate client instance is used for each section so each session block
# is self-contained and can be re-run independently.
client = MultiServerMCPClient(server_config)
async with client.session("agents-tutorial") as session:

    # Static resource: the URI is fixed — every read returns the same server manifest.
    # Clients can use this to discover capabilities without calling list_tools() separately.
    info = await session.read_resource("config://server/info")
    print("==========  config://server/info (static)  ==========")
    print(info.contents[0].text)

    # Template resource: the {category} segment in the URI is resolved at read time.
    # One @mcp.resource("units://reference/{category}") definition on the server covers
    # all variants: /length, /mass, /temperature, /volume — selected by the URI argument.
    units = await session.read_resource("units://reference/mass")
    print("\n==========  units://reference/mass (template)  ==========")
    print(units.contents[0].text)

In [ ]:
client = MultiServerMCPClient(server_config)
async with client.session("agents-tutorial") as session:

    # list_prompts() returns the name and argument schema of every registered prompt.
    available = await session.list_prompts()
    print(f"Prompts available: {[p.name for p in available.prompts]}\n")

    # get_prompt() renders the prompt template with the provided arguments and
    # returns a GetPromptResult containing a list of PromptMessage objects (role + text).
    # The client injects this text as a SystemMessage before running the agent.
    conv = await session.get_prompt("conversion_assistant", arguments={"unit_system": "imperial"})
    print("==========  conversion_assistant (imperial)  ==========")
    print(f"  {conv.messages[0].content.text}")

    tutor = await session.get_prompt("math_tutor", arguments={"level": "advanced"})
    print("\n==========  math_tutor (advanced)  ==========")
    print(f"  {tutor.messages[0].content.text}")

## Configuración del LLM y vinculación de herramientas

In [ ]:
from langchain_core.messages import HumanMessage  
from langchain_ollama import ChatOllama          
# temperature=0 makes tool selection deterministic — the same query always
# triggers the same tool call, which is essential for reproducible tutorials.
llm = ChatOllama(model="qwen2.5:3b", temperature=0)

In [ ]:
client = MultiServerMCPClient(server_config)
tools = await client.get_tools()

# bind_tools() attaches every tool schema to the LLM's context window.
# From this point, every ainvoke() call sends the schemas alongside the messages
# so the model can decide which tool (if any) to call.
llm_with_tools = llm.bind_tools(tools)

In [ ]:
# This call does NOT execute any tool — it asks the LLM to decide which tool
# to call and returns an AIMessage with the tool_calls field populated.
# Actual execution happens later inside tools_node in the agent graph.
probe = await llm_with_tools.ainvoke(
    [HumanMessage(content="I need to convert 50 kg to lbs")]
)

call = probe.tool_calls[0]

In [ ]:
# Each entry in tool_calls is a dict with three keys:
#   "name" — matches the tool registered on the MCP server
#   "args" — arguments the LLM chose to pass
#   "id"   — unique identifier used to pair this call with its ToolMessage response
print(f"Tool selected  : {call['name']}")
print(f"Arguments sent : {call['args']}")

## Construcción del agente con LangGraph

In [ ]:
from langchain_core.messages import SystemMessage, ToolMessage
from typing import TypedDict, Annotated, Literal, Optional
from langchain_mcp_adapters.tools import load_mcp_tools  
from langgraph.graph.message import add_messages          
from langgraph.graph import StateGraph, END
# add_messages is a LangGraph reducer: instead of replacing the messages list on each
# state update, it appends new messages to the existing list.
# This lets agent_node and tools_node each return only the messages they produce;
# LangGraph merges them into a single growing conversation history automatically.
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
def build_graph(llm_with_tools, tools_map):

    """Compile a ReAct agent graph from a bound LLM and a tool dispatch map.

    Graph topology:
        [START] → agent → (tool_calls?) → tools → agent → ... → [END]

    Nodes:
        agent — invokes the LLM; produces an AIMessage with optional tool_calls.
        tools — executes every tool_call in the last AIMessage via the MCP session.

    Edges:
        agent → tools  when the last AIMessage contains at least one tool_call.
        agent → END    when the LLM produces a plain text response (no tool_calls).
        tools → agent  always — returns tool results for the next LLM reasoning step.

    Args:
        llm_with_tools: LLM with MCP tool schemas bound via bind_tools().
        tools_map: {tool_name: BaseTool} for O(1) dispatch during tool execution.

    Returns:
        A compiled CompiledStateGraph ready to be called with ainvoke().
    """

    async def agent_node(state: AgentState):

        # Invoke the LLM with the full conversation history.
        # The response is an AIMessage that may contain tool_calls if the model
        # decided a tool is needed, or plain text if it has the final answer.

        response = await llm_with_tools.ainvoke(state["messages"])
        return {"messages": [response]}

    async def tools_node(state: AgentState):

        # Execute every tool call requested in the last AIMessage.
        # tool_call_id links each ToolMessage back to its originating call so
        # the LLM can correctly attribute results in the next reasoning step.

        last = state["messages"][-1]
        results = []
        for call in last.tool_calls:
            result = await tools_map[call["name"]].ainvoke(call["args"])
            results.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
        return {"messages": results}

    def should_continue(state: AgentState) -> Literal["tools", "__end__"]:

        # Two-step router pattern: agent_node writes tool_calls into state;
        # this function only reads that field to decide the next destination.
        # Keeping the decision here (not inside agent_node) makes the routing
        # independently testable and visible in the graph diagram.
        
        if state["messages"][-1].tool_calls:
            return "tools"
        return END

    graph = StateGraph(AgentState)
    graph.add_node("agent", agent_node)
    graph.add_node("tools", tools_node)
    graph.set_entry_point("agent")
    graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
    graph.add_edge("tools", "agent")   # always return to agent after tool execution

    return graph.compile()

In [ ]:
async def run_agent(query: str, system_prompt: Optional[str] = None) -> str:

    """Open an MCP session, load tools, run the ReAct agent, return the final answer.

    Uses client.session() so the connection to the server stays alive for the full
    duration of the agent execution — including all tool call roundtrips. If the
    session were closed before ainvoke() completes, tool calls would fail mid-graph.

    Args:
        query: The user's question or instruction.
        system_prompt: Optional text injected as a SystemMessage before the query.
                       Pass a server-side prompt here (fetched via get_prompt) to
                       configure the agent's behaviour for a specific task.

    Returns:
        The content of the last message in the conversation — the agent's final answer.
    """

    client = MultiServerMCPClient(server_config)

    async with client.session("agents-tutorial") as session:
        
        # load_mcp_tools wraps the session's tools as LangChain BaseTools.
        # Using the same session for both tool loading and tool execution ensures
        # all calls share a single live subprocess connection.
        tools = await load_mcp_tools(session)
        llm_with_tools = llm.bind_tools(tools)
        tools_map = {t.name: t for t in tools}  # O(1) lookup in tools_node

        app = build_graph(llm_with_tools, tools_map)

        messages = []
        if system_prompt:
            messages.append(SystemMessage(content=system_prompt))
        messages.append(HumanMessage(content=query))

        result = await app.ainvoke({"messages": messages})
        return result["messages"][-1].content
print("build_graph and run_agent defined.")
print("Graph topology: agent -> (tool_calls?) -> tools -> agent -> ... -> END")

## Ejecución de extremo a extremo

### Ejemplo 1: encadenamiento secuencial de herramientas

In [ ]:
# Multi-step tool chaining: the agent must call convert_units first,
# then feed the result into calculate — two sequential MCP tool calls.
print("Query: Convert 100 km to miles, then divide it by 15\n")
answer = await run_agent("Convert 100 km to miles, then divide it by 15")
print(answer)

### Ejemplo 2: herramientas independientes en la misma consulta

In [ ]:
# Two independent tools in the same query: get_weather and calculate
# have no data dependency, so the LLM may call them in any order.
print("Query: What is the weather in Tokyo? Also, what is 2**10 + 5*3?\n")
answer = await run_agent("What is the weather in Tokyo? Also, what is 2**10 + 5*3?")
print(answer)

### Ejemplo 3: inyección de un prompt del servidor

In [ ]:
async def run_with_mcp_prompt(query: str, prompt_name: str, prompt_args: dict) -> str:

    """Run the agent with a system prompt fetched from the MCP server.

    Demonstrates the full MCP client-server loop: resources, prompts, tools,
    and agent output all originate from the same server within a single session.
    The server-defined prompt configures the agent's behaviour before it sees
    the user query — without any prompt text hardcoded on the client side.

    Args:
        query:       The user's question or instruction.
        prompt_name: Name of the server-side prompt to fetch (e.g. "conversion_assistant").
        prompt_args: Arguments passed to the prompt template (e.g. {"unit_system": "metric"}).

    Returns:
        The agent's final answer as a plain string.
    """
    
    client = MultiServerMCPClient(server_config)

    async with client.session("agents-tutorial") as session:
        tools = await load_mcp_tools(session)
        llm_with_tools = llm.bind_tools(tools)
        tools_map = {t.name: t for t in tools}

        # Fetch the server-defined system prompt before building the graph.
        # The prompt text arrives as a PromptMessage; .content.text extracts the string.
        prompt_result = await session.get_prompt(prompt_name, arguments=prompt_args)
        system_text   = prompt_result.messages[0].content.text

        app    = build_graph(llm_with_tools, tools_map)
        result = await app.ainvoke({
            "messages": [
                SystemMessage(content=system_text),  # server-defined behaviour
                HumanMessage(content=query),          # user request
            ]
        })

        return result["messages"][-1].content

In [ ]:
# The conversion_assistant prompt instructs the LLM to show step-by-step workings
# and round to 4 decimal places — behaviour defined on the server, not the client.
print("System prompt: conversion_assistant (metric)\n")
answer = await run_with_mcp_prompt(
    "How many gallons are 15 liters? Show the calculation step by step",
    "conversion_assistant",
    {"unit_system": "metric"},
)
print(answer)